# LeetCode #1294: Weather Type in Each Country

https://leetcode.com/problems/weather-type-in-each-country/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (nested loops in SQL)** | $O(n \cdot m)$ | $O(n)$ |
| **Optimal: Aggregation JOIN ★** | $O(n + m)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force (nested loops in SQL)
Use a correlated subquery to look up the weather for each country row by row. This causes the inner query to execute once per country, leading to $O(n \cdot m)$ I/O.

### Optimal: Aggregation JOIN ★
JOIN the two tables once, then GROUP BY country and compute the average weather state. A single CASE expression maps the average to "Cold", "Hot", or "Warm" using the thresholds from the problem. One pass over both tables with $O(n + m)$ work.

**Constraints:**
* `Countries` has columns: country_id (PK), country_name
* `Day` has columns: country_id (FK), day, weather_state, temperature
* Only November 2019 rows matter (day between 2019-11-01 and 2019-11-30)
* weather_state values: Cold ≤ 15, Warm 16–24, Hot ≥ 25

## Solutions
### C#

In [ ]:
// SQL only — shown as a string for reference
// Aggregate weather_state over November 2019, classify per country
public class Solution {
    public string GetWeatherQuery() => @"
SELECT
    c.country_name,
    CASE
        WHEN AVG(d.weather_state) <= 15 THEN 'Cold'
        WHEN AVG(d.weather_state) >= 25 THEN 'Hot'
        ELSE 'Warm'
    END AS weather_type
FROM Countries c
JOIN Day d ON c.country_id = d.country_id
WHERE d.day BETWEEN '2019-11-01' AND '2019-11-30'
GROUP BY c.country_id, c.country_name;
";
}

### Python

In [ ]:
# SQL only — shown via pandas equivalent for conceptual clarity
import pandas as pd

def weather_type(countries: pd.DataFrame, day: pd.DataFrame) -> pd.DataFrame:
    # Filter to November 2019 only
    nov = day[(day['day'] >= '2019-11-01') & (day['day'] <= '2019-11-30')]
    # Average weather_state per country
    avg_weather = nov.groupby('country_id')['weather_state'].mean().reset_index()
    # Classify using thresholds
    def classify(w):
        if w <= 15:
            return 'Cold'
        elif w >= 25:
            return 'Hot'
        return 'Warm'
    avg_weather['weather_type'] = avg_weather['weather_state'].apply(classify)
    # Join country names
    result = countries.merge(avg_weather[['country_id', 'weather_type']], on='country_id')
    return result[['country_name', 'weather_type']]

### Go

In [ ]:
// SQL only — Go pseudocode showing the aggregation logic
package main

import "fmt"

// WeatherType returns classification given the average weather state
func WeatherType(avgWeather float64) string {
    // Apply thresholds: cold <= 15, hot >= 25, otherwise warm
    switch {
    case avgWeather <= 15:
        return "Cold"
    case avgWeather >= 25:
        return "Hot"
    default:
        return "Warm"
    }
}

func main() {
    // Example: average weather state of 18 is "Warm"
    fmt.Println(WeatherType(18)) // Warm
    fmt.Println(WeatherType(10)) // Cold
    fmt.Println(WeatherType(30)) // Hot
}

### Rust

In [ ]:
// SQL only — Rust pseudocode showing the aggregation logic
fn weather_type(avg_weather: f64) -> &'static str {
    // Apply thresholds: cold <= 15, hot >= 25, otherwise warm
    if avg_weather <= 15.0 {
        "Cold"
    } else if avg_weather >= 25.0 {
        "Hot"
    } else {
        "Warm"
    }
}

fn main() {
    println!("{}", weather_type(18.0)); // Warm
    println!("{}", weather_type(10.0)); // Cold
    println!("{}", weather_type(30.0)); // Hot
}

## Example Scenarios

**1. Common Case** — One country, multiple days all in "Warm" range

**Input:** Country A, November days with weather_state values [18, 20, 17]
Average = $18.3$, which falls in $[16, 24]$. The CASE expression maps this to "Warm".

**2. Slightly Complex** — Two countries, different classifications

**Input:** Country A: states [10, 12, 14] (avg = 12, Cold); Country B: states [26, 28] (avg = 27, Hot)
The JOIN + GROUP BY correctly assigns "Cold" to A and "Hot" to B in a single scan of the November rows.

**3. Edge Case: Time Factor** — Many days per country

**Input:** Country A with all 30 November days, states ranging 1–30
The aggregation sums all 30 values and divides once. Even with large datasets the GROUP BY touches each row exactly once — $O(n)$ where $n$ is the number of Day rows.

**4. Edge Case: Space Factor** — Many countries, single day each

**Input:** 1,000 countries each with one November day entry
The GROUP BY creates 1,000 groups each of size 1. Average equals the single value directly. Output is 1,000 rows — $O(n)$ space in the result set.

**5. Almost-Impossible but Plausible** — Average lands exactly on a boundary

**Input:** Country A: states [15, 16] (avg = 15.5)
The CASE checks `<= 15` first: 15.5 is not $\leq 15$, so it falls to the next branch. 15.5 is not $\geq 25$, so the result is "Warm". Boundary precision matters — integer averages at exactly 15 or 25 map to "Cold" or "Hot" respectively.